In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
import scipy.stats as stats 

### Joint probability

In [2]:
df = sns.load_dataset('titanic')
df.head(3)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True


In [3]:
pd.crosstab(df['survived'],df['pclass'])

pclass,1,2,3
survived,,,
0,80,97,372
1,136,87,119


In [4]:
pd.crosstab(df['survived'],df['pclass'] , normalize=True)

# or 
# pd.crosstab(df['survived'],df['pclass']) / len(df)

pclass,1,2,3
survived,,,
0,0.089787,0.108866,0.417508
1,0.152637,0.097643,0.133558


the `joint probability` of the event -->  person survived and was in pclass 1 is 0.089787  
shows the probability of two events happening together.

### by formula calculation 

`independent events` -  
$$
p\left(A \cap B\right) = p\left(A\right) . p\left(B\right)
$$

In [5]:
# A event --> people who survived 
# B event --> people who was in pclass 1

total = len(df)

joint_probability = len(df[(df['survived'] == 1) & (df['pclass'] == 1)]) / total
joint_probability 

0.1526374859708193

In [6]:
print('Total rows:', len(df))

p_survived = len(df[df['survived'] == 1]) / len(df)
p_pclass = len(df[df['pclass'] == 1]) / len(df)
actual_joint = len(df[(df['survived'] == 1) & (df['pclass'] == 1)]) / len(df)
print('p_survived', p_survived)
print('p_pclass', p_pclass)
print('p_survived * p_pclass', p_survived * p_pclass)
print('actual joint', actual_joint)
print('independent assumption holds?', abs((p_survived * p_pclass) - actual_joint) < 1e-9)

Total rows: 891
p_survived 0.3838383838383838
p_pclass 0.24242424242424243
p_survived * p_pclass 0.09305172941536577
actual joint 0.1526374859708193
independent assumption holds? False


### Marginal probability 

In [7]:
pd.crosstab(df['survived'] , df['pclass'] , margins=True)

pclass,1,2,3,All
survived,,,,
0,80,97,372,549
1,136,87,119,342
All,216,184,491,891


not survived + all -->  549 people died irrespective of their pclass  
survived + all -->  342 people survived irrespective of their pclass  

there are 216 people in pcalss 1 does't matter they dies or survived

In [8]:
pd.crosstab(df['survived'] , df['pclass'] , margins=True, normalize=True)

pclass,1,2,3,All
survived,,,,
0,0.089787,0.108866,0.417508,0.616162
1,0.152637,0.097643,0.133558,0.383838
All,0.242424,0.206510,0.551066,1.000000


there is a probability of 61% or 0.61612 that the people died  
there is a probability of 38% or 0.383838 that the people survived 

In [9]:
# mannual code

# Event --> people survived
survived = df[df['survived'] == 1]
total = len(df)
p_survived = len(survived) / total
print(f'probability of people who survived - {p_survived : .2f}')

probability of people who survived -  0.38


In [10]:
# Event --> people died
died = df[df['survived'] == 0]
total = len(df)
p_died = len(died) / total
print(f'probability of people who survived - {p_died : .2f}')

probability of people who survived -  0.62


### Conditional Probability

In [11]:
pd.crosstab(df.survived, df.pclass , normalize='columns' )

pclass,1,2,3
survived,,,
0,0.37037,0.527174,0.757637
1,0.62963,0.472826,0.242363


prob of surviving or dying given that they are in corresponding pclass
- `normalize = 'column'` calculates the probability of each rows value, given the column category.

In [12]:
# **trick** - normalize by index means index data is given 
pd.crosstab(df.survived, df.pclass , normalize='index' )

pclass,1,2,3
survived,,,
0,0.145719,0.176685,0.677596
1,0.397661,0.254386,0.347953


- prob of being in pclass given that they died or survived
- add of row is 1
- `normalize = 'index'` calculates the probability of each column value, given the row category.

In [13]:
### mannual calculation

# find prob of the people who was traveling in pclass 1 given that they died
# died = len(df[df['survived'] == 0])
died = df[df['survived'] == 0]
total = len(died)
pclass1 = len(died[died['pclass'] == 1])
cond_prob = pclass1 / total
print('prob of the people who was traveling in pclass 1 given that they died : ',cond_prob)

prob of the people who was traveling in pclass 1 given that they died :  0.14571948998178508


`normalize = 'all'` --> joint probability  
`normalize = "all" , margins= True` --> marginal probability  
`normalize = 'column'` or `normalize = 'index'` --> conditional probability

### Baye's Theorem

$$
p\left(A \middle | B\right) = \frac{p\left(B \middle | A\right) \times p\left(A\right)}{p\left(B\right)}
$$

In [14]:
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [16]:
df['sex'].isnull().sum()

np.int64(0)

In [28]:
# calculate the probability of survival given that he is a male . --> will he servive or die.
#  p(survival | M)

# p(M)
males = df[df['sex'] == 'male']
p_males = len(males) / len(df)

# p(Survival)
survived = df[df['survived'] == 1]
p_survived = len(survived) / len(df)

# p(M | survived )
males_given_survived = survived[survived['sex'] == 'male']
p_males_given_survived = len(males_given_survived) / len(survived)

p_survival_given_male = (p_males_given_survived * p_survived) / p_males

print(f"probability of survival given that he is a male is : {p_survival_given_male : .4f}")

print(f"probability of die given that he is a male is : {1 - p_survival_given_male : .4f}")

if p_survival_given_male > (1 - p_survival_given_male) :
    print('he probably will survive ')
else :
    print('he will probably die')


probability of survival given that he is a male is :  0.1889
probability of die given that he is a male is :  0.8111
he will probably die


In [29]:
# Direct calculation (simpler):
males = df[df['sex'] == 'male']
survived_males = len(df[(df['sex'] == 'male') & (df['survived'] == 1)])
p_survival_given_male = survived_males / len(males)

print(f"P(Survival | Male) = {p_survival_given_male:.4f}")

P(Survival | Male) = 0.1889


In [30]:
# Both methods should give same result
print("Method 1 (Bayes' theorem):", p_survival_given_male)
print("Method 2 (Direct):", survived_males / len(males))
print("Results match:", abs((p_survival_given_male) - (survived_males / len(males))) < 1e-10)

Method 1 (Bayes' theorem): 0.18890814558058924
Method 2 (Direct): 0.18890814558058924
Results match: True


In [40]:
pd.crosstab(df['survived'],df['sex'] , normalize='columns' , margins=True)  # prob of survival given male

sex,female,male,All
survived,,,
0,0.257962,0.811092,0.616162
1,0.742038,0.188908,0.383838
